In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data_1 = pd.read_csv("./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
data_2 = pd.read_csv("./data/2023-02-12.csv")

## Préparation des données:

In [ ]:

data_1.columns = data_1.columns.str.replace(" ", "").str.lower()
data_2.columns = data_2.columns.str.replace(" ", "").str.lower()

common_columns = list(set(data_1.columns) & set(data_2.columns))

data_1 = data_1[common_columns]
data_2 = data_2[common_columns]

concatenated_data = pd.concat([data_1, data_2], ignore_index=True)


## Phase 1 : Détection d'Attaques avec Random Forest

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np

# 🔹 Step 1: Prepare Features and Labels
features = [col for col in concatenated_data.columns if col != 'label']
X = concatenated_data[features]
y = concatenated_data['label']

# 🔹 Step 2: Convert Labels to Binary (Attack vs. Normal)
y_binary = np.where(y == "BENIGN", 0, 1)  # 0 = Normal, 1 = Attack

# 🔹 Step 3: Fix NaN & Inf Values in X
X.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert Infs to NaNs
X.fillna(X.mean(), inplace=True)  # Fill NaNs with column mean

# 🔹 Step 4: Normalize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 🔹 Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_binary, test_size=0.2, random_state=42)

# 🔹 Step 6: Apply SMOTE to Balance Classes
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 🔹 Step 7: Train Random Forest for Attack Detection
rf_model = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight='balanced', random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)

# 🔹 Step 8: Evaluate RF
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Performance (Attack Detection):")
print(classification_report(y_test, y_pred_rf))


C:\Users\ibrah\AppData\Local\Temp\ipykernel_37444\3868814369.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert Infs to NaNs
C:\Users\ibrah\AppData\Local\Temp\ipykernel_37444\3868814369.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(X.mean(), inplace=True)  # Fill NaNs with column mean


Random Forest Performance (Attack Detection):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19448
           1       1.00      1.00      1.00     40962

    accuracy                           1.00     60410
   macro avg       1.00      1.00      1.00     60410
weighted avg       1.00      1.00      1.00     60410



## Phase 2: CNN

### Entrainement du CNN

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np

# 🔹 Step 1: Prepare the Features & Labels
features = [col for col in concatenated_data.columns if col != 'label']
X = concatenated_data[features]
y = concatenated_data['label']

# 🔹 Step 2: Handle NaN & Inf Values
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)

# 🔹 Step 3: Normalize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 🔹 Step 4: Encode Labels for Classification
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
y_categorical = to_categorical(y_encoded, num_classes)

# 🔹 Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_categorical, test_size=0.2, random_state=42)

# 🔹 Step 6: Reshape Data for CNN
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 🔹 Step 7: Define CNN Model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    BatchNormalization(),
    Dropout(0.3),

    Conv1D(filters=32, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

# 🔹 Step 8: Compile the Model
model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

# 🔹 Step 9: Train the Model
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# 🔹 Step 10: Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Final CNN Accuracy: {accuracy:.4f}")


C:\Users\ibrah\AppData\Local\Temp\ipykernel_37444\1891128675.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\ibrah\AppData\Local\Temp\ipykernel_37444\1891128675.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(X.mean(), inplace=True)


Epoch 1/5


c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7552/7552 ━━━━━━━━━━━━━━━━━━━━ 55s 7ms/step - accuracy: 0.9778 - loss: 0.0790 - val_accuracy: 0.9940 - val_loss: 0.0195
Epoch 2/5
7552/7552 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.9941 - loss: 0.0204 - val_accuracy: 0.9958 - val_loss: 0.0147
Epoch 3/5
7552/7552 ━━━━━━━━━━━━━━━━━━━━ 55s 7ms/step - accuracy: 0.9945 - loss: 0.0190 - val_accuracy: 0.9958 - val_loss: 0.0142
Epoch 4/5
7552/7552 ━━━━━━━━━━━━━━━━━━━━ 57s 8ms/step - accuracy: 0.9949 - loss: 0.0171 - val_accuracy: 0.9959 - val_loss: 0.0148
Epoch 5/5
7552/7552 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - accuracy: 0.9949 - loss: 0.0169 - val_accuracy: 0.9961 - val_loss: 0.0140
1888/1888 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9963 - loss: 0.0127
Final CNN Accuracy: 0.9961


## Fusion des deux modèle (Random Forest && CNN)

In [30]:
import pandas as pd
import numpy as np

# 🔹 Step 1: Predict Attack/Normal with RF
X_test_rf = X_test.reshape(X_test.shape[0], X_test.shape[1])  # Ensure it's 2D
rf_predictions = rf_model.predict(X_test_rf)

# 🔹 Step 2: Select Only Attack Samples for CNN
attack_indices = np.where(rf_predictions == 1)[0]  # Indices where RF predicted "Attack"
X_test_attacks = X_test[attack_indices]  # Only attack samples

# 🔹 Step 3: Prepare Attack Data for CNN
X_test_attacks = X_test_attacks.reshape(X_test_attacks.shape[0], X_test_attacks.shape[1])  # Convert to 2D
X_test_attacks = scaler.transform(X_test_attacks)  # Normalize
X_test_attacks = X_test_attacks.reshape(X_test_attacks.shape[0], X_test_attacks.shape[1], 1)  # Convert back to 3D

# 🔹 Step 4: Predict Attack Type with CNN
cnn_predictions = model.predict(X_test_attacks)
cnn_final_preds = np.argmax(cnn_predictions, axis=1)  # Get attack type indices

# 🔹 Step 5: Convert Attack Type Indices to Labels
cnn_labels = label_encoder.inverse_transform(cnn_final_preds)

# 🔹 Step 6: Merge RF & CNN Results
final_predictions = np.array(["BENIGN"] * len(X_test))  # Start with all "Normal"
final_predictions[attack_indices] = cnn_labels  # Replace attack predictions with CNN output

# 🔹 Step 7: Evaluate Final Hybrid Model
X_test_df = pd.DataFrame(X_test_rf, columns=features)  # Convert X_test to DataFrame

# 🔹 Use X_test_df instead of X_test
y_test_labels = concatenated_data.iloc[X_test_df.index]['label'].values  
print("Hybrid Model Performance (Attack Detection & Type Classification):")
print(classification_report(y_test_labels, final_predictions))


   1/1280 ━━━━━━━━━━━━━━━━━━━━ 2:03 96ms/step

c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Hybrid Model Performance (Attack Detection & Type Classification):
              precision    recall  f1-score   support

      BENIGN       0.50      0.99      0.66     30012
        DDoS       0.00      0.00      0.00     30398
      cowrie       0.00      0.00      0.00         0
      log4po       0.00      0.00      0.00         0

    accuracy                           0.49     60410
   macro avg       0.12      0.25      0.17     60410
weighted avg       0.25      0.49      0.33     60410



c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is"